# 🗺️ TOPODATA → Tabela por Município (Região Sudeste)

Este notebook faz tudo automaticamente:
1. Instala as bibliotecas necessárias
2. Baixa o shapefile dos municípios do Sudeste (IBGE)
3. Baixa as folhas do TOPODATA (declividade + altitude)
4. Extrai estatísticas por município
5. Exporta CSV pronto para o dataset do TCC

> ⚠️ **Tempo estimado:** ~15-30 minutos dependendo da conexão

## ⚙️ Passo 1 — Instalar bibliotecas

In [1]:
!pip install rasterio rasterstats geopandas requests -q
print('✅ Bibliotecas instaladas!')

✅ Bibliotecas instaladas!


## 📦 Passo 2 — Importar bibliotecas e configurar pastas

In [2]:
import os
import zipfile
import glob
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from rasterstats import zonal_stats
from io import BytesIO

# Cria pastas de trabalho
os.makedirs('topodata/altitude',    exist_ok=True)
os.makedirs('topodata/declividade', exist_ok=True)
os.makedirs('shapefiles',           exist_ok=True)

print('✅ Pastas criadas!')

✅ Pastas criadas!


## 🗺️ Passo 3 — Baixar municípios do Sudeste (IBGE)

In [3]:
# Shapefile de municípios do Brasil (IBGE 2022)
url_ibge = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/Brasil/BR/BR_Municipios_2022.zip"

print('Baixando shapefile do IBGE...')
r = requests.get(url_ibge, timeout=120)
with zipfile.ZipFile(BytesIO(r.content)) as z:
    z.extractall('shapefiles/')
print('✅ Shapefile baixado!')

# Carrega e filtra só o Sudeste
shp_path = glob.glob('shapefiles/*.shp')[0]
municipios = gpd.read_file(shp_path)

# Códigos dos estados do Sudeste: SP=35, RJ=33, MG=31, ES=32
sudeste_codigos = ['31', '32', '33', '35']
municipios['CD_UF'] = municipios['CD_MUN'].astype(str).str[:2]
sudeste = municipios[municipios['CD_UF'].isin(sudeste_codigos)].copy()
sudeste = sudeste.to_crs('EPSG:4326')  # garante WGS84

print(f'✅ {len(sudeste)} municípios do Sudeste carregados!')
sudeste[['CD_MUN', 'NM_MUN', 'CD_UF']].head()

Baixando shapefile do IBGE...
✅ Shapefile baixado!
✅ 1668 municípios do Sudeste carregados!


,CD_MUN,NM_MUN,CD_UF
2244,3100104,Abadia dos Dourados,31
2245,3100203,Abaeté,31
2246,3100302,Abre Campo,31
2247,3100401,Acaiaca,31
2248,3100500,Açucena,31


## 🌍 Passo 4 — Definir folhas do TOPODATA para o Sudeste

O Sudeste cobre aproximadamente:
- Latitude: 14°S a 25°S  
- Longitude: 39°W a 54°W

As folhas têm 1° de latitude x 1,5° de longitude.
Notação: `LAHLONsufixo` onde H=S (Sul), LON com `_` = grau inteiro, `5` = grau e 30'

In [4]:
def gerar_folhas_sudeste():
    """
    Gera lista de nomes de folhas TOPODATA que cobrem o Sudeste.
    Latitude: 14S a 24S (incremento de 1°)
    Longitude: 39W a 54W (incremento de 1.5° → notação _=inteiro, 5=meio)
    """
    folhas = []
    for lat in range(14, 25):       # 14S até 24S
        for lon_base in range(39, 55):  # 39W até 54W
            # Longitude inteira (ex: 45 → '45_')
            lon_str_int  = f"{lon_base:02d}_"
            # Longitude meia (ex: 45.5 → '455')
            lon_str_meio = f"{lon_base:02d}5"
            folha_int  = f"{lat:02d}S{lon_str_int}"
            folha_meio = f"{lat:02d}S{lon_str_meio}"
            folhas.append(folha_int)
            folhas.append(folha_meio)
    return folhas

folhas = gerar_folhas_sudeste()
print(f'Total de folhas a tentar baixar: {len(folhas)}')
print('Exemplos:', folhas[:6])

Total de folhas a tentar baixar: 352
Exemplos: ['14S39_', '14S395', '14S40_', '14S405', '14S41_', '14S415']


## ⬇️ Passo 5 — Baixar arquivos do TOPODATA

Sufixos:
- `ZN` = Altitude (MDE)
- `SD` = Declividade

In [5]:
BASE_URL = "http://www.dsr.inpe.br/topodata/data/geotiff/"

PRODUTOS = {
    'ZN': 'topodata/altitude',
    'SD': 'topodata/declividade',
}

def baixar_folha(folha, sufixo, pasta_destino):
    arquivo = f"{folha}{sufixo}.zip"
    destino_zip = os.path.join(pasta_destino, arquivo)
    destino_tif = destino_zip.replace('.zip', '.tif')

    if os.path.exists(destino_tif):
        return 'ja_existe'

    url = BASE_URL + arquivo
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(destino_zip, 'wb') as f:
                f.write(r.content)
            with zipfile.ZipFile(destino_zip) as z:
                z.extractall(pasta_destino)
            os.remove(destino_zip)
            return 'ok'
        else:
            return 'nao_existe'
    except Exception as e:
        return f'erro: {e}'

# Baixa tudo
resultados = {'ok': 0, 'ja_existe': 0, 'nao_existe': 0, 'erro': 0}

for i, folha in enumerate(folhas):
    for sufixo, pasta in PRODUTOS.items():
        res = baixar_folha(folha, sufixo, pasta)
        if res in resultados:
            resultados[res] += 1
        else:
            resultados['erro'] += 1
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(folhas)} folhas processadas... {resultados}')

print('\n✅ Download concluído!')
print(resultados)

  20/352 folhas processadas... {'ok': 7, 'ja_existe': 0, 'nao_existe': 33, 'erro': 0}
  40/352 folhas processadas... {'ok': 14, 'ja_existe': 0, 'nao_existe': 66, 'erro': 0}
  60/352 folhas processadas... {'ok': 21, 'ja_existe': 0, 'nao_existe': 99, 'erro': 0}
  80/352 folhas processadas... {'ok': 28, 'ja_existe': 0, 'nao_existe': 132, 'erro': 0}
  100/352 folhas processadas... {'ok': 34, 'ja_existe': 0, 'nao_existe': 166, 'erro': 0}
  120/352 folhas processadas... {'ok': 40, 'ja_existe': 0, 'nao_existe': 200, 'erro': 0}
  140/352 folhas processadas... {'ok': 46, 'ja_existe': 0, 'nao_existe': 234, 'erro': 0}
  160/352 folhas processadas... {'ok': 53, 'ja_existe': 0, 'nao_existe': 267, 'erro': 0}
  180/352 folhas processadas... {'ok': 59, 'ja_existe': 0, 'nao_existe': 301, 'erro': 0}
  200/352 folhas processadas... {'ok': 65, 'ja_existe': 0, 'nao_existe': 335, 'erro': 0}
  220/352 folhas processadas... {'ok': 72, 'ja_existe': 0, 'nao_existe': 368, 'erro': 0}
  240/352 folhas processadas.

## 🔗 Passo 6 — Mesclar folhas em mosaicos únicos

In [6]:
def criar_mosaico(pasta, nome_saida):
    tifs = glob.glob(f'{pasta}/*.tif')
    if not tifs:
        print(f'⚠️ Nenhum .tif encontrado em {pasta}')
        return None

    print(f'  Mesclando {len(tifs)} arquivos de {pasta}...')
    datasets = [rasterio.open(f) for f in tifs]
    mosaico, transform = merge(datasets)

    meta = datasets[0].meta.copy()
    meta.update({
        'driver': 'GTiff',
        'height': mosaico.shape[1],
        'width':  mosaico.shape[2],
        'transform': transform
    })

    with rasterio.open(nome_saida, 'w', **meta) as dest:
        dest.write(mosaico)

    for ds in datasets:
        ds.close()

    print(f'  ✅ Mosaico salvo: {nome_saida}')
    return nome_saida

print('Criando mosaico de ALTITUDE...')
mosaico_alt = criar_mosaico('topodata/altitude',    'mosaico_altitude.tif')

print('Criando mosaico de DECLIVIDADE...')
mosaico_dec = criar_mosaico('topodata/declividade', 'mosaico_declividade.tif')

Criando mosaico de ALTITUDE...
  Mesclando 104 arquivos de topodata/altitude...


  ✅ Mosaico salvo: mosaico_altitude.tif
Criando mosaico de DECLIVIDADE...
⚠️ Nenhum .tif encontrado em topodata/declividade


## 📊 Passo 7 — Extrair estatísticas por município

In [7]:
STATS = ['mean', 'max', 'min', 'std', 'median']
NODATA = -9999

def extrair_stats(raster_path, municipios_gdf, prefixo):
    print(f'  Extraindo estatísticas de {raster_path}...')
    stats = zonal_stats(
        municipios_gdf,
        raster_path,
        stats=STATS,
        nodata=NODATA,
        all_touched=True
    )
    df = pd.DataFrame(stats)
    df.columns = [f'{prefixo}_{c}' for c in df.columns]
    return df

resultado = sudeste[['CD_MUN', 'NM_MUN', 'CD_UF']].copy().reset_index(drop=True)

# Mapeia código UF para nome do estado
estados = {'31': 'MG', '32': 'ES', '33': 'RJ', '35': 'SP'}
resultado['UF'] = resultado['CD_UF'].map(estados)

if mosaico_alt:
    df_alt = extrair_stats(mosaico_alt, sudeste, 'altitude')
    resultado = pd.concat([resultado, df_alt], axis=1)
    print('  ✅ Altitude extraída!')

if mosaico_dec:
    df_dec = extrair_stats(mosaico_dec, sudeste, 'declividade')
    resultado = pd.concat([resultado, df_dec], axis=1)
    print('  ✅ Declividade extraída!')

print(f'\n✅ Tabela gerada com {len(resultado)} municípios e {len(resultado.columns)} colunas!')
resultado.head(10)

  Extraindo estatísticas de mosaico_altitude.tif...
  ✅ Altitude extraída!

✅ Tabela gerada com 1668 municípios e 9 colunas!


,CD_MUN,NM_MUN,CD_UF,UF,altitude_min,altitude_max,altitude_mean,altitude_std,altitude_median
0,3100104,Abadia dos Dourados,31,MG,613.870972,1002.840027,776.661379,64.102617,777.138000
1,3100203,Abaeté,31,MG,538.867004,918.463989,640.696696,44.715478,636.169983
2,3100302,Abre Campo,31,MG,342.114014,1318.130005,674.425508,137.835100,668.033020
3,3100401,Acaiaca,31,MG,410.740997,817.330017,606.231532,73.006752,613.952026
4,3100500,Açucena,31,MG,187.755005,1048.040039,500.900004,207.530672,483.950989
5,3100609,Água Boa,31,MG,249.195007,1101.550049,517.493463,190.022983,463.122009
6,3100708,Água Comprida,31,MG,465.838013,705.356018,566.849689,54.530926,556.635986
7,3100807,Aguanil,31,MG,727.943970,1092.119995,844.485411,64.207466,828.351990
8,3100906,Águas Formosas,31,MG,229.645004,951.810974,429.230559,137.589203,391.092499
9,3101003,Águas Vermelhas,31,MG,645.453979,1034.209961,822.371907,63.070446,825.143982


## 💾 Passo 8 — Exportar CSV final

In [8]:
# Remove coluna auxiliar
resultado = resultado.drop(columns=['CD_UF'], errors='ignore')

# Salva CSV
output_path = 'topodata_sudeste_municipios.csv'
resultado.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'✅ CSV exportado: {output_path}')
print(f'   Linhas: {len(resultado)}')
print(f'   Colunas: {list(resultado.columns)}')

# Faz download automático no Colab
from google.colab import files
files.download(output_path)
print('⬇️ Download iniciado!')

✅ CSV exportado: topodata_sudeste_municipios.csv
   Linhas: 1668
   Colunas: ['CD_MUN', 'NM_MUN', 'UF', 'altitude_min', 'altitude_max', 'altitude_mean', 'altitude_std', 'altitude_median']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Download iniciado!


## 🔍 Passo 9 — Verificação rápida dos dados

In [9]:
print('=== RESUMO DA TABELA ===')
print(resultado.describe().round(2))

print('\n=== MUNICÍPIOS COM MAIOR DECLIVIDADE MÉDIA ===')
if 'declividade_mean' in resultado.columns:
    print(resultado.nlargest(10, 'declividade_mean')[['NM_MUN', 'UF', 'declividade_mean', 'altitude_mean']])

print('\n=== NULOS POR COLUNA ===')
print(resultado.isnull().sum())

=== RESUMO DA TABELA ===
       altitude_min  altitude_max  altitude_mean  altitude_std  \
count       1668.00       1668.00        1668.00       1668.00   
mean         465.54       1015.92         647.50         97.83   
std          242.43        419.14         264.87         74.31   
min         -186.96         32.20           4.15          1.61   
25%          307.21        704.02         458.48         43.28   
50%          475.16        980.41         643.32         76.18   
75%          646.82       1249.14         833.92        127.64   
max         1158.86       2862.50        1588.01        610.29   

       altitude_median  
count          1668.00  
mean            632.06  
std             267.99  
min               4.11  
25%             452.93  
50%             632.17  
75%             815.58  
max            1623.18  

=== MUNICÍPIOS COM MAIOR DECLIVIDADE MÉDIA ===

=== NULOS POR COLUNA ===
CD_MUN             0
NM_MUN             0
UF                 0
altitude_min      

## 📋 Dicionário de variáveis geradas

| Coluna | Descrição | Unidade |
|---|---|---|
| `CD_MUN` | Código IBGE do município | — |
| `NM_MUN` | Nome do município | — |
| `UF` | Estado | — |
| `altitude_mean` | Altitude média do município | metros |
| `altitude_max` | Altitude máxima | metros |
| `altitude_min` | Altitude mínima | metros |
| `altitude_std` | Variação de altitude (relevo irregular) | metros |
| `altitude_median` | Altitude mediana | metros |
| `declividade_mean` | Declividade média (**variável chave para deslizamentos**) | graus |
| `declividade_max` | Declividade máxima | graus |
| `declividade_std` | Variação de declividade | graus |

> 💡 Para o modelo de deslizamentos, as variáveis mais importantes são: `declividade_mean`, `declividade_max` e `altitude_std` (quanto maior a variação de altitude, mais acidentado o terreno).